In [1]:
# Load Dependencies
import numpy as np
import pandas as pd
from itertools import product
from scipy.interpolate import interp1d
import itertools
from itertools import product
import sys
import re
import statistics
#sys.path.append('/Users/meoskin/Jupyter')

BASE_THRESHOLD = 0.95
HIATUS_THRESHOLD = 0.975
CORRELATION_THRESHOLD = 0.05
MINIMUM_EVENT_QUALITY = 1
MINIMUM_JOINT_PROBABILITY = 1e-10

In [2]:
# Define Earthquake Event class [and associated helper functions]
class Earthquake:

    #Constructor: stores earthquake name, age, PDF, and CDF for each earthquake event (i.e. Bidart.Event_2)
    def __init__(self, name, age, PDF, CDF):
        self.name = name
        self.age = np.array(age)
        self.PDF = np.array(PDF)
        self.CDF = np.array(CDF)
    #Returns single sampled age based on PDF: generate random number between 0 and 1, then sample the age from the CDF that corresponds to number
    def event(self):
        r = np.random.rand()
        return self.age[np.argmin(np.abs(self.CDF - r))]
    #Returns n sampled ages: calls event() n times, then stores results in an array
    def events(self, n):
        return np.array([self.event() for _ in range(n)])
    #Resamples earthquake age to yearly resolution: calculates age range (L), then creates list of years inside age range and interpolates event PDF and CDF to those years
    def resample(self):
        L = int(self.age[-1] - self.age[0])
        new_age = np.linspace(self.age[0], self.age[-1], L + 1)

        f_age = interp1d(self.age, self.age, fill_value="extrapolate")
        f_pdf = interp1d(self.age, self.PDF, fill_value="extrapolate")
        f_cdf = interp1d(self.age, self.CDF, fill_value="extrapolate")

        self.age = f_age(new_age)
        self.PDF = f_pdf(new_age)
        self.PDF = self.PDF / np.sum(self.PDF)
        self.CDF = f_cdf(new_age)

    
    def BC(self,other):
        # Make sure both objects have the same sampling interval
        diff_age = np.mean(np.diff(self.age))
        if diff_age != np.mean(np.diff(other.age)):
            raise ValueError("Sampling interval differs")

        # Make sure both objects sample the same times of the year
        misalign = self.age[0] % 1 - other.age[0] % 1
        if not np.isclose(misalign,0): # if the array of ages are on different times of the year, shift so they allign so that BC can be calculated (only basal boundary ages fall on whole-number years)
        # shift the two distributions so they line up
            self_age = self.age.copy()
            self_age = self_age + misalign
            self_PDF = np.interp(self_age, self.age, self.PDF)
            # calculate Bhattacharya Coefficient
            this_df = pd.DataFrame({'age':self_age, 'this_pdf':self_PDF})
            that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
            overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
            overlap_df['root_product'] = np.sqrt(overlap_df['this_pdf'] * overlap_df['that_pdf'])
            return(sum(overlap_df['root_product']))
            
        else: #If distributions are not misaligned
            # calculate Bhattacharya Coefficient
            this_df = pd.DataFrame({'age':self.age, 'this_pdf':self.PDF})
            that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
            overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
            overlap_df['root_product'] = np.sqrt(overlap_df['this_pdf'] * overlap_df['that_pdf'])
            return(sum(overlap_df['root_product']))
                             
    
    #Calculates joint probability of multiple earthquake events: find age spacing, and make sure both events have same spacing
    def jointprobability(self, other,name='Joint'):
        # Make sure both objects have the same sampling interval
        diff_age = np.mean(np.diff(self.age))
        if diff_age != np.mean(np.diff(other.age)):
            raise ValueError("Sampling interval differs")

        # Make sure both objects sample the same times of the year
        misalign = self.age[0] % 1 - other.age[0] % 1
        if not np.isclose(misalign,0): # if the array of ages are on different times of the year, shift so they allign so that BC can be calculated (only basal boundary ages fall on whole-number years)
        # shift the two distributions so they line up
            self_age = self.age.copy()
            self_age = self_age + misalign
            self_PDF = np.interp(self_age, self.age, self.PDF)
            # calculate joint probability
            this_df = pd.DataFrame({'age':self_age, 'this_pdf':self_PDF})
            that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
            if this_df.empty or that_df.empty: raise ValueError('Warning: One or more dataframes empty, check age allignment!')
            overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
            overlap_df['joint_pdf'] = overlap_df['this_pdf'] * overlap_df['that_pdf']
            total_joint_probability = sum(overlap_df['joint_pdf'])
            if total_joint_probability < MINIMUM_JOINT_PROBABILITY: raise RuntimeError('Sum joint probability close to zero!')
            overlap_df['joint_pdf'] = overlap_df['joint_pdf']/total_joint_probability
            overlap_df['joint_cdf'] = np.cumsum(overlap_df['joint_pdf'])
            return Earthquake(name, overlap_df['age'],overlap_df['joint_pdf'],overlap_df['joint_cdf'])
            
        else: #If distributions are not misaligned
            # calculate joint probability
            this_df = pd.DataFrame({'age':self.age, 'this_pdf':self.PDF})
            that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
            overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
            overlap_df['joint_pdf'] = overlap_df['this_pdf'] * overlap_df['that_pdf']
            total_joint_probability = sum(overlap_df['joint_pdf'])
            if total_joint_probability < MINIMUM_JOINT_PROBABILITY: raise RuntimeError('Sum joint probability close to zero!')
            overlap_df['joint_pdf'] = overlap_df['joint_pdf']/total_joint_probability
            overlap_df['joint_cdf'] = np.cumsum(overlap_df['joint_pdf'])
            return Earthquake(name, overlap_df['age'],overlap_df['joint_pdf'],overlap_df['joint_cdf'])


class Candidate_Event(Earthquake):
    def __init__(self, name, event_list=[]):
        self.name = name
        self.event_list = event_list
        joint = event_list[next((i for i, x in enumerate(event_list) if x.name.split('_', 1)[-1] != 'basal_boundary'), None)] # find the index of the first non-basal boundary object in the candidate
        for event in event_list[1:len(event_list)]:
            if event.name.split('_', 1)[-1] == 'basal_boundary': # don't calculate joint probability with basal boundaries
                continue
            else:
                joint = joint.jointprobability(event)
        self.age = list(joint.age)
        self.PDF = list(joint.PDF)
        self.CDF = list(joint.CDF)

In [3]:
# Define paleoseismic dataset object
class paleoseismic_dataset:
    # constructor stores essential site, fault, and position information
    def __init__(self,site_name,fault_name,position,slip_threshold = 0):
        self.site_name = site_name
        self.fault_name = fault_name
        self.position = position
        self.slip_threshold = slip_threshold
        self.exclude = []
        self.event_names = []

    def load_oxcal(self,filename, name_selector='Event', base_selector='Base', 
                   hiatus_selector = 'Hiatus', delim=None):
        if delim:
            df = pd.read_csv(filename, sep=delim)
        else:
            df = pd.read_csv(filename)
    
        # Keep only posterior type
        df = df[df["type"] == "posterior"]

        # Subset for events and extract

        df_events = df[df["name"].str.contains(name_selector, na=False)]

        self.events = []

        for eventname, group in df_events.groupby("name", sort=False): 
            eventname_fixed = re.sub(r"\s+", "_", eventname)
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]
    
            self.events.append(
                Earthquake(
                    name=self.site_name + "_" + eventname_fixed,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )

        # Extract event names and produce generic event quality tuple:
        for event in self.events:
            self.event_names.append(event.name)
        self.event_quality = dict(zip(self.event_names, [4] * len(self.events))) # set default event quality to 4

        # Subset for basal boundary
        df_base = df[df["name"].str.contains(base_selector, na=False)]
        ages = df_base["value"].values
        pdf = df_base["probability"].values
        cdf = np.cumsum(pdf)
        self.basal_boundary = ages[np.searchsorted(cdf, 1 - BASE_THRESHOLD)]
##ERIK EXPERIMENTING WITH STUFF
        self.base = []
        self.base = Earthquake(name=self.site_name+"_basal_boundary",age=ages,PDF=pdf,CDF=cdf)
##

        # Subset hiatuses
        df_hiatus = df[df["name"].str.contains(hiatus_selector, na=False)]
        self.hiatuses = []

        hiatusnumber = 1
        for hiatus, group in df_hiatus.groupby("name", sort=False):
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]

            self.hiatuses.append(
                Earthquake(
                    name=self.site_name + "_hiatus_"+str(hiatusnumber),
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
            hiatusnumber = hiatusnumber + 1
                
    def get_marginal_pdfs(self,minimum_event_quality = 1):
        rows = []
        for i, eq in enumerate(self.events):
            if eq.name in self.exclude:
                continue
            if self.event_quality[eq.name] < minimum_event_quality:
                continue
            df = pd.DataFrame({
                "event": self.site_name + "_" + eq.name,
                "year": eq.age,
                "prob": eq.PDF,
                "CDF": eq.CDF
                
            })
            #df = df[df["year"] > 0]
            df["site_order"] = i
            rows.append(df)
        return rows

    def event_list(self):
        output_list = []
        for i, eq in enumerate(self.events):
            if eq.name in self.exclude:
                continue
            if self.event_quality[eq.name] < MINIMUM_EVENT_QUALITY:
                continue
            output_list.append(eq)
        return(output_list)

In [4]:
# Annette Paleoseismic Site

Annette_location = 20 # km
Annette = paleoseismic_dataset("Annette","San Andreas", Annette_location)
Annette.load_oxcal("C:/Users/erikc/OneDrive/Documents/Research/Annette_inverted.csv")
Annette.exclude = ['Annette_Groundshaking_Event_']
Annette.event_quality.update(Annette_Event_2 = 3, Annette_Event_3 = 4, Annette_Event_4 = 5, Annette_Event_5 = 4, Annette_Event_6 = 5)

In [5]:
# Bidart Paleoseismic Site

Bidart_location = 72 # km
Bidart = paleoseismic_dataset("Bidart", "San Andreas", Bidart_location)
Bidart.load_oxcal("C:/Users/erikc/Downloads/Bidart_Akciz2023_edited1 (1).csv")
Bidart.exclude = ['Bidart_Event_1']
Bidart.event_quality.update(Bidart_Event_2 = 5, Bidart_Event_3 = 5, Bidart_Event_4 = 5, Bidart_Event_5 = 5, Bidart_Event_6 = 5)

In [6]:
# Frazier Mountain Paleoseismic Site
Frazier_location = 169 # km
Frazier = paleoseismic_dataset("Frazier","San Andreas", Frazier_location)
Frazier.load_oxcal("C:/Users/erikc/Downloads/FrazierMountain_HiatusesIncluded.csv")
Frazier.exclude = ['Frazier_Event_Historic_1857', 'Frazier_=Event_11'] #This weird 'extra' event comes from the new way we model hiatuses at the end of the OxCal script with a Sum()
#Frazier.event_quality.update(Frazier
Frazier.event_quality.update(Frazier_Event_2 = 5, Frazier_Event_3 = 5, Frazier_Event_4 = 5, Frazier_Event_5 = 5, Frazier_Event_6 = 3, Frazier_Event_7 = 5, Frazier_Event_8 = 4, Frazier_Event_9 = 5, Frazier_Event_10 = 3, Frazier_Event_11 = 4)
#print(Frazier.get_event_list())
#Frazier.get_marginal_pdfs(excluded_events)
#Frazier.event_quality
#Frazier.hiatuses[0].age#[np.where(Frazier.hiatuses[0].CDF > 0.1)[0]]

In [7]:
#Elizabeth Lake Paleoseismic Site

Elizabeth_location = 218 # km
Elizabeth = paleoseismic_dataset("Elizabeth", "San Andreas", Elizabeth_location)
Elizabeth.load_oxcal("C:/Users/erikc/Downloads/Elizabeth_HiatusesIncluded.csv")
Elizabeth.event_quality.update(Elizabeth_Event_EL2 = 3, Elizabeth_Event_EL3 = 3, Elizabeth_Event_EL4 = 3, Elizabeth_Event_ELX = 2)
#Elizabeth.hiatuses[1].age[np.where(Elizabeth.hiatuses[0].CDF > 0.1)[1]]
#Elizabeth.hiatuses[1].age
#Elizabeth.hiatuses[1].age

In [8]:
# Pallett Creek Paleoseismic Site

Pallett_location = 270 # km
Pallett = paleoseismic_dataset("Pallett","San Andreas", Pallett_location)
Pallett.load_oxcal("C:/Users/erikc/Downloads/PallettCreek_HiatusesIncluded.csv")
#print(Pallett.event_names)
Pallett.exclude = ['Pallett_Event_X', 'Pallett_Event_MRE']
Pallett.event_quality.update(Pallett_Event_V = 5, Pallett_Event_T_and_Hiatus = 5, Pallett_Event_R = 5, Pallett_Event_N = 4, Pallett_Event_I = 4, Pallett_Event_F_and_Hiatus = 5, Pallett_Event_D = 4, Pallett_Event_C = 4)
#Pallett.event_quality
#Pallett.get_marginal_pdfs(4)
#Pallett.hiatuses[0].age

In [9]:
#Wrightwood Paleoseismic Site

Wrightwood_location = 292 # km
Wrightwood = paleoseismic_dataset("Wrightwood", "San Andreas", Wrightwood_location)
Wrightwood.load_oxcal("C:/Users/erikc/Downloads/Wrightwood_HiatusesIncluded.csv")
print(Wrightwood.event_names)
Wrightwood.exclude = ['Wrightwood_Event_Historic_1812', 'Wrightwood_Event_Historic_1857']
Wrightwood.event_quality.update(Wrightwood_Event_3 = 5, Wrightwood_Event_4 = 5, Wrightwood_Event_5 = 5, Wrightwood_Event_6 = 5, Wrightwood_Event_7 = 4, Wrightwood_Event_8 = 4, Wrightwood_Event_9 = 4, Wrightwood_Event_10 = 4, Wrightwood_Event_11 = 5, Wrightwood_Event_12 = 4, Wrightwood_Event_13 = 3, Wrightwood_Event_14 = 4)
#Wrightwood.hiatuses[0].age[np.where(Wrightwood.hiatuses[0].CDF < 0.95)[0]]
#Wrightwood.hiatuses[1].age[np.where(Wrightwood.hiatuses[1].CDF < 0.95)[0]]

['Wrightwood_Event_W14', 'Wrightwood_Event_W13', 'Wrightwood_Event_W12', 'Wrightwood_Event_W11', 'Wrightwood_Event_W10', 'Wrightwood_Event_W9', 'Wrightwood_Event_W8', 'Wrightwood_Event_W7', 'Wrightwood_Event_W6', 'Wrightwood_Event_W5', 'Wrightwood_Event_W4', 'Wrightwood_Event_W3', 'Wrightwood_Event_Historic_1812', 'Wrightwood_Event_Historic_1857']


In [10]:
#Pitman Canyon Paleoseismic Site

Pitman_location = 317 # km
Pitman = paleoseismic_dataset("Pitman", "San Andreas", Pitman_location)
Pitman.load_oxcal("C:/Users/erikc/OneDrive/Documents/Research/PitmanCanyon.csv")

In [11]:
#Burro Flat Paleoseismic Site

Burro_location = 377 # km
Burro = paleoseismic_dataset("Burro", "San Andreas", Burro_location)
Burro.load_oxcal("C:/Users/erikc/OneDrive/Documents/Research/BurroFlats.csv")
Burro.event_quality.update(Burro_Event_3 = 5)

In [12]:
#Cabezon Paleoseismic Site

Cabezon_location = #To be continued after I finish setting up the new Event Enumeration and build up the fault in CalTopo through San Gorgonio Pass

SyntaxError: invalid syntax (4282371563.py, line 3)

In [12]:
##

In [13]:
#Erik experimenting with stuff: make correlation matrix including hiatuses and basal boundaries

# Set up master event list for enumeration of candidates

included_sites = [Annette, Bidart, Frazier, Elizabeth, Pallett, Wrightwood, Pitman, Burro]
included_site_names = [site.site_name for site in included_sites]

LEFT_END = 0
LEFT_END_MINSLIP = 0
RIGHT_END = 390
RIGHT_END_MINSLIP = 0

master_event_list = [] # earthquake objects
master_site_list = [] # paleoseismic site objects in fault-position order
master_object_list = [] # earthquake, hiatus, and basal boundary objects
master_site_list = [] # paleoseismic site objects in fault-position order

for site in included_sites:
    master_event_list = master_event_list + site.event_list()
    master_object_list = master_object_list + site.event_list() + site.hiatuses + [site.base]
    master_site_list = master_site_list + [site] * len(site.event_list()) + [site] * len(site.hiatuses) + [site] # extra + [site] at the end accounts for the basal boundary (since it has no length attribute and len(base) = 1

master_event_names = [eq.name for eq in master_event_list] # corresponding list of event names
master_site_names = [site.site_name for site in master_site_list] # corresponding list of event names
master_object_names = [obj.name for obj in master_object_list]

#master_event_names.index(Pallett.events[7].name) Example of how to find index of an event in matrix

#total_events = len(master_event_names)
total_objects = len(master_object_list)

# Construct correlation matrix
correlation_matrix = np.zeros((total_objects,total_objects))

for i in range(total_objects):
    for j in range(total_objects):
        if master_object_list[i].name.split('_', 1)[-1] == 'basal_boundary' and master_object_list[j].name.split('_', 1)[-1] == 'basal_boundary': # we have chosen to automatically correlate basal boundaries: this should not effect event enumeration in next step
            correlation_matrix[i,j] = 1        
        elif master_object_list[i].name.split('_', 1)[-1] == 'basal_boundary' or master_object_list[j].name.split('_', 1)[-1] == 'basal_boundary': # if we are correlating an event or hiatus and a basal boundary:
            idx = [master_object_list[i].name.split('_', 1)[-1], master_object_list[j].name.split('_', 1)[-1]].index('basal_boundary')
            boundary_object = [master_object_list[i], master_object_list[j]][idx] # save the '.base' object
            other_object = [master_object_list[i], master_object_list[j]][abs(idx - 1)] # save the other object
            # test if that event (or hiatus) correlate with the no - record period at that trench (basal boundary):
            if np.average(other_object.age, weights = other_object.PDF) > np.average(boundary_object.age, weights = boundary_object.PDF): # if mean(event.age) or mean(hiatus.age) younger than mean(base.age):
                # check bhattacharya coefficient of the two pdfs to decide wether to correlate
                correlation_matrix[i,j] = other_object.BC(boundary_object)
            else:
                correlation_matrix[i,j] = 1 #Set to highest possible BC, assuming that the mean of the event PDF being older than the mean of the basal boundary PDF means that the event/hiatus has to be older than the trench record
        else: #If we are only correlating events and/or hiatuses:
            correlation_matrix[i,j] = master_object_list[i].BC(master_object_list[j]) # works for hiatuses as well

In [14]:
# Recursive candidate enumeration algorithm
candidate_events = []
RIGHT = 1
LEFT = 2
def make_candidate(correlation_row, new_object, direction = RIGHT, left_objects = [], right_objects = [], right_end = None, right_minslip = None):
    # correlation_row: one row of the correlation matrix
    # root: event at the root of this candidate tree. All events correlate to root by definition.
    # left_events: list of events in the left dirction already added to the candidate
    # right_events: list of events in the right direction already added to the candidate
    # new_event: proposed new event at the next trench site
    # direction: are we adding an event to the RIGHT or LEFT?
    
    # STEP 1: Check new event against all right and left events to make sure that this candidate is valid.
    # If not valid, then return, otherwise continue and make event and recursively call as needed.
    new_object_index = master_object_names.index(new_object.name)
    combined_objects = left_objects + right_objects
    for object in combined_objects:
        object_index = master_object_names.index(object.name)
        # Test if new event is valid correlation to each other event in the list (except root).
        # If an invalid correlation is found, we return (end the function) and don't generate a candidate 
        if correlation_matrix[new_object_index,object_index] < CORRELATION_THRESHOLD: return

    # STEP 2. Look for the next possible event (one or more) and call make_candidate for each of these. This
    # step recursively builds the candidates. To ensure against infinite recursion:
    # If the direction == RIGHT, then we look to the right and then to the left
    # If the direction == LEFT, then we only look to the left
    if direction == RIGHT:
        # find name of rightmost site
        rightmost_site_name = master_site_names[master_object_names.index(new_object.name)]
        # find the name of the next site or end of fault
        rightmost_site_index = included_site_names.index(rightmost_site_name)
        if rightmost_site_index == len(included_site_names) - 1:
            # We found the end of the fault, use this as the end point
            right_end = RIGHT_END
            right_minslip = RIGHT_END_MINSLIP
        else:
            # We look at the next trench site on the list and call next level of recursion if we find possible correlation
            next_trench_site = included_sites[rightmost_site_index + 1]
            next_trench_site_name = next_trench_site.site_name
            possible_correlative_object_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
            for object_index in possible_correlative_object_indices:
                # This is the key step in the recursion. If an event at the next trench has a possible correlation,
                # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
                # If more than one possible correlative is found make_candidate is called for each.
                if correlation_row[object_index] >= CORRELATION_THRESHOLD:
                    next_new_object = master_object_list[object_index]
                    make_candidate(correlation_row, next_new_object, RIGHT, left_objects, right_objects + [new_object])
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
            right_end = next_trench_site.position
            right_minslip = next_trench_site.slip_threshold

        # Final steps, looking to the right:
        right_objects = right_objects + [new_object] # Copy new event to right event list. List complete for this candidate.
        new_object = right_objects[0] # new_event is now the root event so we can start recursion to the left
        
    # Finished looking to right, now look left
    
    leftmost_site_name = master_site_names[master_object_names.index(new_object.name)]
    # find the name of the next site or end of fault
    leftmost_site_index = included_site_names.index(leftmost_site_name)
    if leftmost_site_index == 0:
            # We found the end of the fault, use this as the other end point
        left_end = LEFT_END
        left_minslip = LEFT_END_MINSLIP
    else:
        # We look at the next trench site on the list and call next level of recursion if we find possible correlation
        next_trench_site = included_sites[leftmost_site_index - 1]
        next_trench_site_name = next_trench_site.site_name
        possible_correlative_object_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
        for object_index in possible_correlative_object_indices:
            # This is the key step in the recursion. If an event at the next trench has a possible correlation,
            # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
            # If more than one possible correlative is foumd make_candidate is called for each.
            if correlation_row[object_index] >= CORRELATION_THRESHOLD:
                next_new_object = master_object_list[object_index]
                make_candidate(correlation_row, next_new_object, LEFT, left_objects + [new_object], right_objects, right_end, right_minslip)
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
        left_end = next_trench_site.position
        left_minslip = next_trench_site.slip_threshold
    
    # Final steps, looking to the left:
    left_objects = left_objects + [new_object] # Copy new event to left event list. List complete for this candidate.
    left_objects = left_objects[1:] # Remove the root event left_events (it is preserved in right_events)
    temporary_event_name = right_objects[0].name # name this event after the root event. Elaborate names later....

    # Final step: create candidate and add it to the list
    candidate_events.append(Candidate_Event(temporary_event_name, left_objects[::-1] + right_objects))
    candidate_events[-1].right_end = right_end
    candidate_events[-1].right_minslip = right_minslip
    candidate_events[-1].left_end = left_end
    candidate_events[-1].left_minslip = left_minslip

In [15]:
# build master candidate event list, only using events as roots

candidate_events = []

for event in master_event_names:
    object_index = master_object_names.index(event)
    make_candidate(correlation_matrix[object_index], master_object_list[object_index])

In [16]:
# convert candidates to matrix (IMPORTANT CHOICE: unless suggested otherwise, we will build the candidates_matrix and candidates_df using ONLY events)

candidates_matrix = np.zeros((len(candidate_events), len(master_event_list)))

for i, candidate in enumerate(candidate_events):
    for event in candidate.event_list:
        if event.name in master_event_names:
            idx = master_event_names.index(event.name)
            candidates_matrix[i][idx] = 1

# convert matrix to dataframe with extra 'event_id' column, drop duplicate candidates

candidates_df = pd.DataFrame(candidates_matrix, columns=master_event_names)
candidates_df["event_id"] = np.arange(len(candidates_df))
candidates_df.drop_duplicates(subset = master_event_names, inplace=True) # drop duplicates inplace, checking all columns except 'event_id', because 'event_id' would make each row unique

# remove corresponding duplicate candidates from candidate_events based on 'event_id'

candidate_events = [item for i, item in enumerate(candidate_events) if i in candidates_df['event_id']]

# now remove event_id column to avoid accidentally creating constraints for it in ILP Model

candidates_df.drop(columns = 'event_id').reset_index(drop=True)

,Annette_Event_6,Annette_Event_5,Annette_Event_4,Annette_Event_3,Annette_Event_2,Bidart_Event_6,Bidart_Event_5,Bidart_Event_4,Bidart_Event_3,Bidart_Event_2,...,Pitman_Event_3,Pitman_Event_2,Burro_Event_9,Burro_Event_8,Burro_Event_7,Burro_Event_6,Burro_Event_5,Burro_Event_4,Burro_Event_3,Burro_Event_2
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
819,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
820,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
821,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [17]:
# import dependencies

import math
import pandas as pd
import itertools as it
import pulp as pl
import matplotlib.pyplot as plt
import random
import kuehn_et_al_fdm as kea
from kuehn_et_al_fdm import calc_displ_site
from kuehn_et_al_fdm import calc_displ_avg
import gurobipy as gp # NOTE: will have to download Guroby optimize software from their site
import re

In [18]:
#Save a reference array that contains the mean displacements for each magnitude from 6.5 to 8 with resolution down to 0.01 magnitude differences
#Used for magnitude calculation later that is then input into the Kuehn et. al FDM

magnitudes = np.arange(6.5,8.01,0.01)
rupture_lengths = np.arange(6.5,8.01,0.01)
mean_displacement_array = np.arange(6.5, 8.01, 0.01)
for i in range(len(magnitudes)):
    this_magnitude = magnitudes[i]
    mean_displacement = kea.calc_displ_avg(
    magnitude = this_magnitude,
    style = 'strike-slip',
    coefficient_type = 'mean'
)
    mean_displacement_array[i] = mean_displacement
    rupture_lengths[i] = ((10**(1.5*this_magnitude + 9.105))/((3*10**10)*(mean_displacement)*15000))

#Store variables used in calculation of magnitude from surface rupture length (from Lavrentiadis et. al, 2023)
B1 = -1.1602
B2 = 0.3395
Wlim = 15 #seismogenic depth for vertically dipping fault with 15km seismogenic thickness - derivation is Dseis / sin(theta) where Dseis = seismogenic thickness and theta = dip angle
Fd = 0 #For San Andreas Fault - see 'A seismologically consistent surface rupture length model for unbounded and width-limited events', Lavrentiadis et. al, 2023
c0 = -3.38783
c1 = 0.9509
c2 = -0.1517

#Detection Threshold Code (WARNING: 'calculated' is experimental (based on Biasi and Weldon's theoretical workflow) and not trusted)

#Calculate or input thresholds for detection_probability_threshold % chance of detection based on displacement, for each site

#NOTE: Depending on the threshold, this may make model infeasible (more susceptible at sites with high observation bounds, i.e. Annette, Elizabeth Lake)

displacement_threshold_setting = 'estimated'

threshold = []

if displacement_threshold_setting == 'estimated':

    threshold = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5] #Manually input thresholds here

if displacement_threshold_setting == 'calculated':
    detection_probability_threshold = 50#% (percentage probability of being detected) (percentiles)
#If site has no available spatial coverage factor and/or structural relief factor, put 'na' in for site's position in both arrays, and an empty list in for stratigraphic thicknesses
    spatial_coverage_factor = [1, 1, 1, 1, 1, 1, 1, 1, 0.66, 'na', 'na', 'na']

    structural_relief_factor = [23.75, 11.5, 7, 16, 13, 1.8, 8.6, 10, 2.2, 'na', 'na', 'na']

    stratigraphic_thicknesses = [[16, 8, 16, 8, 5, 7, 64, 8, 7, 5, 7, 5, 8, 9, 12, 27, 8, 16, 16, 10, 7, 5, 4, 5, 7, 16, 4, 5, 4, 10, 8, 5, 4, 4, 5, 8, 7, 5, 5, 25, 5, 3, 3, 3, 3, 4, 32, 19, 8, 4, 5, 4, 5, 3, 3, 3, 5, 4, 4, 4, 5, 2.7, 4, 3, 3, 3, 5, 7, 3, 8, 3, 7, 5, 8, 8, 4, 5.3, 19, 8, 56, 37],
                             [24, 20, 5, 24, 27, 15, 27, 29, 7, 7, 15, 49, 17, 22, 7, 7, 15, 15, 17, 17],
                             [12, 25, 9, 11, 16, 16, 19, 5, 28, 9, 16, 4, 2, 3, 4, 5, 4, 4, 2, 23, 4, 4, 5, 7, 7, 18, 4, 4, 5, 4, 23, 11, 2, 4, 21, 14, 1, 12, 7, 4, 4, 4, 2, 4, 14, 7, 11, 12, 5, 25, 9, 7, 2, 5, 11, 40, 4, 5, 7, 35, 2, 4, 23, 11, 35, 11, 14],
                             [75, 30, 7.5, 10, 27.5, 27.5, 7.5, 10, 12.5, 35, 20, 20, 15, 7.5],
                             [9, 11, 1, 11, 1, 11, 1, 5, 7, 4, 11, 48, 53, 95, 11, 44, 1, 23, 1, 12, 9, 16, 1, 68, 12, 5, 7, 4, 1, 33, 1, 14, 11, 33, 30, 4, 4, 4, 1, 12, 18, 14, 1, 28, 9, 12, 14, 9, 23],
                             [5, 5, 5, 9, 59, 5, 45, 2, 29, 4, 5, 2, 29, 4, 105, 4, 15, 4, 33, 4, 76, 4, 64, 4, 53, 27, 2, 9, 4, 9, 4, 22, 7, 11, 5, 27, 5, 4, 64, 13, 25, 18, 4, 11, 16, 4, 7, 13, 16, 15, 7, 20, 5, 18, 11, 7, 4, 15, 2, 22, 20, 24, 35, 24, 15, 15, 11, 15],
                             [6, 13, 6, 9, 6, 9, 6, 19, 94, 13, 16, 6, 6, 44, 6, 38, 9, 25, 13, 63, 19, 19, 28, 13, 50, 6, 9, 3, 38, 41, 19, 3, 19, 19, 25, 13, 19, 9, 25],
                             [61, 20, 61, 9, 23, 56, 11, 9, 9, 41, 5, 11, 7, 5, 43, 9, 5, 32, 45, 5, 36, 13.5, 45, 14],
                             [20, 80, 130, 120, 50, 100, 50, 130, 70, 80, 90, 60, 80, 50, 130, 90, 100, 100],
                             [],
                             [],
                             []
                            ]
    for i in range(len(stratigraphic_thicknesses)):
        if spatial_coverage_factor[i] == 'na' and structural_relief_factor[i] == 'na': #If site has no available estimates of spatial coverage and structural relief, use default of 0.5 m
            threshold.append(0.5)
        else:
            actual_probability = detection_probability_threshold/spatial_coverage_factor[i]
            adjusted_stratigraphic_thicknesses = structural_relief_factor[i]*np.array(stratigraphic_thicknesses[i])
            threshold.append((np.percentile(adjusted_stratigraphic_thicknesses, actual_probability))/100)

In [19]:
#Create dataframes for slip rate measurements and slip per event measurements

##IMPORTANT CHOICE: I have chosen to model up to 1812, since that is the oldest historical earthquake. 1812 and
#1857 slip are subtracted off of each slip rate site from the original author's measurement, based on Zielke et al.
#paper. HOWEVER this will tend to lower slip rates for sites that are used as a constant slip rate
#(i.e. Wallace Creek, Gorman Creek, Ranch Center) and should be tested for its effect on the model output

#Create dictionary that holds site name, location, displacement lower bound, displacement upper bound, age lower and upper bounds,
#oldest and youngest possible ages of lower age bound, oldest and youngest possible ages of upper age bound

#IMPORTANT CHOICE: Added 1 m uncertainties on Bidart Fan offset measurements and all measurements where only a single value was given for constraint purposes

###Big Bend lower default slip rate above 10 mm/yr is making model infeasible: for now set to 9 mm/yr

#Historic slip (1812 + 1857 combined) subtracted off for every SRS, estimated via geomorphology by Zielke et al. at https://pubs.geoscienceworld.org/ssa/bssa/article/102/3/1135/349677/High-Resolution-Topography-Derived-Offsets-along
#(except for Bidart Fan, which was estimated by the SRS investigation, but probably replace with Zielke et al. estimate and add back slip to Bidart Fan)

#[3.5, 5.9, 4.5, 4.2, 4, 1.9, 3, 2.4, 3.28 - 3.67*, 2.6 - 2.7*, 1.35 - 1.43*]
#*Estimated using Kuehn et al. model as no quantitative slip estimates for 1812 available at these sites

#IMPORTANT NOTE: Decided to add 1.2 m of slip to upper bound of Bidart Fan (young) displacement measurement, based on discrepency between slip measured by
#Zielke et al. there in 1857 (4.7 m) vs. the slip Sara Grant Ludwig et al. estimated there (5.9 m) in 1857

#IMPORTANT NOTE: Removed SPE for Frazier Mountain, after Kate said it was a bad idea there

SRS_dictionary = {
    "name": ["Wallace Creek", "Bidart Fan (old)", "Bidart Fan (young)", "Van Matre Ranch", "Gorman Creek", "Oak Flat", "Ranch Center", "Wrightwood", "Cajon Creek", "Pitman Canyon", "Plunge Creek", "Badger Canyon"],
    "location": [66, 72, 72, 84, 176, 198, 238, 292, 314, 348, 317, 374], 
    "displacement mean" : [130, 5.7, 4.3, 20.3, 70.8, 39, 74.6, 42, 287.6, 646.5, 397.4, 283.6],
    "displacement lower bound": [125.5, 4.7, 3.3, 19.3, 65.8, 36, 56.1, 27, 277.6, 421.5, 397.4, 238.6],
    "displacement upper bound": [127.5, 6.7, 6.5, 21.3, 75.8, 42, 93.1, 57, 317.6, 946.5, 507.4, 298.6],
    "age lower bound": [-1561, 1418, 1605, 1218, -925, 915, 0, 464, -10588, -43350, -33050, -21150],
    "age upper bound": [1812, 1605, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812],
    "oldest lower bound": [-1614, 1408, 1600, 1160, -1055, 780, -26, 464, -11588, -45950, -34510, -21550],
    "youngest lower bound": [-1508, 1428, 1610, 1276, -812, 1050, 26, 464, -9588, -40650, -30220, -20750],
    "oldest upper bound": [1812, 1600, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812],
    "youngest upper bound": [1812, 1610, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812, 1812],
}

SRS_df = pd.DataFrame(SRS_dictionary)

#*IMPORTANT NOTE: Disagreement in 1812 + 1857 slip in Wrightwood vicinity. Salyards et al. (1992) estimated
#8.5 +- 1 m total combined slip at Pallett Creek, whereas Zielke et al. (2012) estimated 3.1 m total combined slip at Wrightwood
#WILL NEED TO TEST RANGE OF VALUES BETWEEN 3.1 AND 9.5 m

#**NOTE: slip rate sites whose upper bound is present have had historical slip (1812 and 1857) subtracted off their total displacement

#Wrightwood is probably missing an event between 1300 and 1400, per https://rock.geosociety.org/net/gsatoday/archive/14/9/pdf/i1052-5173-14-9-4.pdf (also good source for 1812 displacements)

#Wallace Creek should be used as a lower and upper bound (for range of values from 35 - 37mm/yr)
#Bidart Fan displacement sets should be used to optimize against, 
#Gorman Creek should be used as a lower bound, 
#Oak Flat should be used as a lower and upper bound to constrain (34 - 51mm/yr), 
#Ranch Center should be used as a lower and upper bound to constrain
#Pallett Creek displacement sets should be used to optimize against
#Wrightwood should be optimized against
#Plunge Creek should be used as a lower and upper bound
#Badger Canyon should be used to optimize against
#Pitman Canyon should be used to optimize against
#Cajon Creek should be used to optimize against
#Van Matre Ranch should be used to optimize against

SRS_site_names = np.array(SRS_df["name"])

#Input names of sites you want to OPTIMIZE WITH here:
optimize_site_names = ["Bidart Fan (old)", "Bidart Fan (young)", "Van Matre Ranch", "Wrightwood", "Badger Canyon", "Pitman Canyon", "Cajon Creek"]

#Input names of sites you want to CONSTRAIN (LOWER AND UPPER BOUND) WITH here:
constrain_site_names = ["Van Matre Ranch", "Wallace Creek", "Bidart Fan (old)", "Bidart Fan (young)", "Oak Flat", "Ranch Center", "Wrightwood", "Plunge Creek", "Badger Canyon", "Pitman Canyon", "Cajon Creek", "Van Matre Ranch"]

#Input names of sites you want to CONSTRAIN (LOWER BOUND) WITH here:
constrain_lower_bd_site_names = ["Gorman Creek"]

#Input names of sites you want to CONSTRAIN (UPPER BOUND) WITH here:
constrain_upper_bd_site_names = []

##Use after extending fault model
#Set default slip rate constraints at intervals along full length of fault

##Test: lower and raise default slip rates
Big_Bend_default_low_rate = 0.009 #20 - 51 mm/yr for 'Big Bend' section, based on SCEC Geologic slip rate minimum and maximum rates along section
Big_Bend_default_high_rate = 0.051

San_Bernadino_default_low_rate = 0.002 #2 - 28 mm/yr for 'San Bernadino' section (i.e. between Cajon and San Gorgonio passes), again based on the maximum and minimum rates in the SCEC Slip Rate Database
San_Bernadino_default_high_rate = 0.028

#Slip-per-event data: constrain model based on estimates of single-event slip from trenches

#Frazier_vertical_displacements = [0.8, 0.85, 0.2, 1, 0.25 - 0.6, 0.27, 0.48, 0.27, 0 - 1, 0 - 1]
#Pallett_brittle_displacements = [0.8 - 1.5*, 1 - 1.9*, 0.5 - 1, 0.1 - 0.2, 0.1 - 0.15, 2, 1 - 3, 1 - 3]  

# *summed slip from these first two events has to be between 2.5m - 2.7m
#Pallett Event X not included because it is the 1812 historical San Andreas earthquake (set sum of decision variables in associated candidates table == 0)
#Pallett_Event_I upper bound raised from 0.73 to 1.2 m since it was making model unsolvable in SPE_constrain mode

#Using scalar factor calculated from vertical to horizontal ratio for Fort Tejon Earthquake
#for Annette site yields MINIMUM slip rate of 69 mm/yr; therefore, I do not include SPE for Annette site

#Choose events whose summed displacement is better constrained
summed_d_events = [['Pallett_Event_V', 'Pallett_Event_T_and_Hiatus'], ['Burro_Event_3', 'Burro_Event_4'], ['Pitman_Event_2', 'Pitman_Event_3', 'Pitman_Event_4']]
summed_d_mean = [[6.9], [8], [12]]
summed_d_lower_bd = [[1.5], [7], [10]]
summed_d_upper_bd = [[10.5], [9], [14]]

SPE_dictionary = {
    "event": [#'Frazier_Event_2', 'Frazier_Event_3', 'Frazier_Event_4', 'Frazier_Event_5', 'Frazier_Event_6', 'Frazier_Event_7', 'Frazier_Event_8', 'Frazier_Event_9', 'Frazier_Event_10', 'Frazier_Event_11',
              'Pallett_Event_V', 'Pallett_Event_T_and_Hiatus', 'Pallett_Event_R', 'Pallett_Event_N', 'Pallett_Event_I', 'Pallett_Event_F_and_Hiatus', 'Pallett_Event_D', 'Pallett_Event_C',
              'Wrightwood_Event_W3', 'Wrightwood_Event_W4', 'Wrightwood_Event_W5', 'Wrightwood_Event_W6', 'Wrightwood_Event_W7', 'Wrightwood_Event_W8', 'Wrightwood_Event_W9', 'Wrightwood_Event_W10', 'Wrightwood_Event_W11', 'Wrightwood_Event_W12', 'Wrightwood_Event_W13', 'Wrightwood_Event_W14',
              'Pitman_Event_2',
              'Burro_Event_2'],
             
    "lower_bd": [#2.2, 2.4, 0.56, 2.8, 0.7, 0.75, 1.3, 0.75, 0.56, 0.56,
                 2.5, 1.60, 0.80, 0.16, 0.16, 3.2, 1.6, 1.6,
                 1.0, 3.0, 0.0, 1.9, 1.1, 0.7, 3.0, 2.5, 1.2, 1.4, 1.0, 1.0,
                 3.0,
                 3.0],
                
    "upper_bd": [#7.9, 8.4, 2.0, 9.9, 5.9, 2.7, 4.8, 2.7, 9.9, 9.9,
                 8.5, 9.3, 4.9, 0.98, 1.2, 9.8, 15, 15,
                 7.0, 9.0, 2.8, 5.6, 3.4, 3.1, 9.9, 7.5, 6.3, 8.2, 5.1, 3.8,
                 5.0,
                 5.0],
                
    "mean": [#4.9, 5.2, 1.2, 6.1, 2.6, 1.7, 2.9, 1.7, 3.1, 3.1,
             5.5, 4.7, 2.9, 0.55, 0.47, 7.3, 7.3, 7.3, 
             3.5, 7.0, 0.7, 3.7, 1.8, 1.5, 6.6, 5.2, 3.0, 4.1, 1.8, 1.9,
             4.0,
             4.0]
            
}

SPE_df = pd.DataFrame(SPE_dictionary)

#Used Frazier Mountain SPE to estimate dextral slip per event (using best fitting scalar factor between vertical and
#horizontal slip of 6.14 based on Zielke and Arrowsmith's 1857
#Fort Tejon Earthquake slip profile giving ~4.3m of slip at Frazier Mountain, which caused an observed 0.7m of
#vertical slip at Frazier Mountain to get a slip rate of ~25.9 mm/yr, maximum scalar factor of 9.90 to get a slip rate of 51 mm/yr,
#minimum scalar factor of 2.78 to get a slip rate of 10 mm/yr)

#Used Pallett Creek SPE to estimate total (brittle + nonbrittle) dextral slip per event (using best fitting scalar factor of
#2.64 based on Salyard and Sieh's estimate of 14 m of nonbrittle deformation caused by ~5.3 m of brittle deformation
#during 3 most recent earthquakes at Pallett Creek (Events Z, X, V) to get a slip rate of 30.6 mm/yr (using mean SPEs)), 
#maximum scalar factor of 3.88 to get a slip rate of 51 mm/yr, minimum scalar factor of 0.601 to get a slip rate of 10 mm/yr


#Put events in order of increasing size from left -> right
SPE_relative_list = [
    ['Annette_Event_2', 'Annette_Event_3', 'Annette_Event_6', 'Annette_Event_4', 'Annette_Event_5'],
    ['Bidart_Event_3', 'Bidart_Event_4'],
    ['Frazier_Event_4', 'Frazier_Event_7', 'Frazier_Event_6', 'Frazier_Event_8', 'Frazier_Event_10', 'Frazier_Event_2', 'Frazier_Event_3', 'Frazier_Event_5'],
    ['Pallett_Event_I', 'Pallett_Event_N', 'Pallett_Event_R', 'Pallett_Event_T_and_Hiatus', 'Pallett_Event_V'],
    ['Wrightwood_Event_W5', 'Wrightwood_Event_W8', 'Wrightwood_Event_W14', 'Wrightwood_Event_W11', 'Wrightwood_Event_W3', 'Wrightwood_Event_W6', 'Wrightwood_Event_W12', 'Wrightwood_Event_W10', 'Wrightwood_Event_W9', 'Wrightwood_Event_W4'],
    ['Bidart_Event_4', 'Bidart_Event_3', 'Bidart_Event_6', 'Bidart_Event_2', 'Bidart_Event_5'] #NOTE: These relative size estimates comes from Wallace Creek instead of Bidart, per this study: https://pubs.geoscienceworld.org/gsa/geology/article/32/8/649/29500/Six-similar-sequential-ruptures-of-the-San-Andreas
]

#Put same size events in their own bracketed ([ ]) groups
equal_size_events = [['Frazier_Event_7', 'Frazier_Event_9'], ['Frazier_Event_10', 'Frazier_Event_11'], ['Pallett_Event_C', 'Pallett_Event_D', 'Pallett_Event_F_and_Hiatus'], ['Wrightwood_Event_W7', 'Wrightwood_Event_W13']]

In [20]:
t = 15 #t km endpoint intervals

l = 20 # km: Used to allow ruptures to extend past the first and last apparently unruptured trench: 20km is arbitrary value to make sure ruptures can extend through apparently 'unruptured' trenches: inequality for unruptured trenches will later filter out endpoints that would cause too much displacement at 'unruptured' trenches

default_slip_rate_constraint_spacing = 40 #Spacing of points (km) to constrain absolute lower and upper bound of possible slip rate

SRL_threshold = 45 #Minimum surface rupture length (km) to possibly create an observable displacement (NOTE: should be 47 km according to Kuehn et al. model, but doing so makes all single-site events at Pallett Creek have 0 displacement due to spacing of endpoints and proximity to other trenches)

fault_length = RIGHT_END

#Precalculate all possible displacements (1 km resolution) for all possible endpoint spacing (to 't' km resolution) for the given fault_length
#Used for fast creation of the displacements matrix

precalculated_displacements = [[] for _ in range(math.ceil((fault_length + 2*l)/t))] #Use fault_length+2l rounded up to the nearest integer because ruptures are allowed to extend l km past end of fault on either side in creation of N and S possible rupture endpoints

for i, rupturelength in enumerate(range(0, fault_length + 2*l, t)):
    for j in range(rupturelength):
        index = (np.abs(rupture_lengths/1000 - (rupturelength))).argmin()
        magnitude = (index)*0.01 + 6.5
        displacement = kea.calc_displ_site(
            magnitude = magnitude,
            location = j/(rupturelength), #In order of origin of rupture (hypocenter) at j = 0 to end of rupture at j = rupturelength
            style = 'strike-slip', 
            percentile = -1 #KNOB: using mean (-1) for percentile, will later test different percentiles of slip
        )
        precalculated_displacements[i].append(displacement)


#Create displacement matrices for each trench (to be used to constrain observed/unobserved displacements at each site
#as well as slip-per-event (SPE) optimization and constraints)

trench_displacements = [[[] for i in range(len(included_sites))] for j in range(len(candidate_events))]

for x in range(len(included_sites)):                     #loop "columnwise", for each paleoseismic site individually
    for n in range(len(candidate_events)):              #loop for each candidate event
        Possible_N_Endpoints_Array = np.arange(candidate_events[n].left_end - l, candidate_events[n].right_end + l, t)
        Possible_S_Endpoints_Array = np.arange(candidate_events[n].left_end - l, candidate_events[n].right_end + l, t)
        for i in range(len(Possible_S_Endpoints_Array)):  #Loop over all possible S Endpoints based on ruptured/unruptured constraints
            for j in range(i): #Loop over all possible N Endpoints for given S Endpoint based on S > N constraint
                #If the trench location is outside the rupture OR the rupture length is smaller than M6.5, set to 0
                if included_sites[x].position >= Possible_S_Endpoints_Array[i] or included_sites[x].position <= Possible_N_Endpoints_Array[j] or Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j] < SRL_threshold:
                    Displacement = 0
                    trench_displacements[n][x].append(Displacement)
                else:
                    rupture_l = Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j]
                    index = (np.abs(rupture_lengths/1000 - (rupture_l))).argmin()
                    magnitude = (index)*0.01 + 6.5
                    y = rupture_l/t #Index of row of precalculated_displacements that corresponds to this rupture length
                    z = Possible_S_Endpoints_Array[i] - included_sites[x].position #KNOB: Assumes earthquakes are propagating northwards
                    Displacement = precalculated_displacements[int(y)][z]
                    trench_displacements[n][x].append(Displacement)

#Create displacements matrix for each slip rate site for each candidate event

srs_displacements = [[[] for i in range(len(SRS_df['name']))] for j in range(len(candidate_events))]

for x in range(len(SRS_df['name'])):                     #loop "columnwise", for each paleoseismic site individually
    for n in range(len(candidate_events)):              #loop for each candidate event
        Possible_N_Endpoints_Array = np.arange(candidate_events[n].left_end - l, candidate_events[n].right_end + l, t)
        Possible_S_Endpoints_Array = np.arange(candidate_events[n].left_end - l, candidate_events[n].right_end + l, t)
        for i in range(len(Possible_S_Endpoints_Array)):  #Loop over all possible S Endpoints based on ruptured/unruptred constraints
            for j in range(i):                            #Loop over all possible N Endpoints for given S Endpoint based on S > N constraint
                if SRS_df['location'][x] >= Possible_S_Endpoints_Array[i] or SRS_df['location'][x] <= Possible_N_Endpoints_Array[j] or Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j] < SRL_threshold:
                    Displacement = 0
                    srs_displacements[n][x].append(Displacement)
                else:
                    rupture_l = Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j]
                    index = (np.abs(rupture_lengths/1000 - (rupture_l))).argmin()
                    magnitude = (index)*0.01 + 6.5
                    y = rupture_l/t #Index of row of precalculated_displacements that corresponds to this rupture length
                    z = Possible_S_Endpoints_Array[i] - SRS_df['location'][x] #KNOB: Assumes earthquakes are propagating northwards
                    Displacement = precalculated_displacements[int(y)][z]
                    srs_displacements[n][x].append(Displacement)

#Create displacements matrix for points uniformly spaced along fault to constrain slip rate

default_slip_rate_locations = np.arange(included_sites[0].position, included_sites[-1].position, default_slip_rate_constraint_spacing)

default_displacements = [[[] for i in range(len(default_slip_rate_locations))] for j in range(len(candidate_events))]

for x in range(len(default_slip_rate_locations)):                     #loop "columnwise", for each paleoseismic site individually
    for n in range(len(candidate_events)):              #loop for each candidate event
        Possible_N_Endpoints_Array = np.arange(candidate_events[n].left_end - l, candidate_events[n].right_end + l, t)
        Possible_S_Endpoints_Array = np.arange(candidate_events[n].left_end - l, candidate_events[n].right_end + l, t)
        for i in range(len(Possible_S_Endpoints_Array)):  #Loop over all possible S Endpoints based on ruptured/unruptred constraints
            for j in range(i):                            #Loop over all possible N Endpoints for given S Endpoint based on S > N constraint
                if default_slip_rate_locations[x] >= Possible_S_Endpoints_Array[i] or default_slip_rate_locations[x] <= Possible_N_Endpoints_Array[j] or Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j] < SRL_threshold:
                    Displacement = 0
                    default_displacements[n][x].append(Displacement)
                else:
                    rupture_l = Possible_S_Endpoints_Array[i] - Possible_N_Endpoints_Array[j]
                    index = (np.abs(rupture_lengths/1000 - (rupture_l))).argmin()
                    magnitude = (index)*0.01 + 6.5
                    y = rupture_l/t #Index of row of precalculated_displacements that corresponds to this rupture length
                    z = Possible_S_Endpoints_Array[i] - default_slip_rate_locations[x] #KNOB: Assumes earthquakes are propagating northwards
                    Displacement = precalculated_displacements[int(y)][z]
                    default_displacements[n][x].append(Displacement)
#Precompute indices of ruptured vs. unruptured trenches for each candidate event
#Also precompute indices of displacements from 'displacements' that are above and below threshold (currently set to 0.5m for observation)
#Make displacements array uniform so all values are 0D numpy arrays


#Might not be necessary: convert all values in site binary matrix to be floats?
#for n in range(len(site_binary_matrix)):
#    for site in range(len(site_binary_matrix[0])):
#        for i in range(len(trench_displacements[n][site])):
#            val = trench_displacements[n][site][i]
#            if isinstance(val, np.ndarray):
#                trench_displacements[n][site][i] = float(val[0])

#Store endpoints used to calculate displacements in their own seperate arrays (to be indexed later to find actual endpoint values once decision variables are chosen)
#Follows same matrix-derived pattern as above, but only one set (one array) for each candidate event (each site displacement matrix is calculated using the same matrix of endpoints)

S = [[] for n in range(len(candidate_events))]
N = [[] for n in range(len(candidate_events))]


for n in range(len(candidate_events)):  
    for i in range(int((candidate_events[n].right_end + l - candidate_events[n].left_end - l)/t)+1):
        for j in range(i):
            endpoint = candidate_events[n].left_end - l + t*i
            S[n].append(endpoint)

for n in range(len(candidate_events)):  
    for i in range(int((candidate_events[n].right_end + l - candidate_events[n].left_end - l)/t)+1):
        for j in range(i):
            endpoint = candidate_events[n].left_end - l + t*j
            N[n].append(endpoint)

#Store the total number of possible endpoint combinations generated for each candidate event in its own array
candidate_displacements_count = []
for n in range(len(candidate_events)):
    count = len(trench_displacements[n][0])
    candidate_displacements_count.append(count)

In [ ]:
##

In [21]:
trench_displacements[0][4]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 array([1.59837864]),
 array([1.57652414]),
 array([1.53719034]),
 array([1.51965469]),
 array([1.48562729]),
 array([1.45479654]),
 array([1.42749255]),
 array([1.4041623]),
 array([1.3854192]),
 array([1.35445118]),
 array([1.33031117]),


In [ ]:
##

In [24]:
#Precalculate constraints for ILP Model

CDF_stratigraphic_constraint_threshold = 0.44 #(From 0 to 0.5) The two percentiles, x and 1 - x, that you want to use to define the highest probability time interval of your event, for use in constraining that these percentiles of each trench event are in stratigraphic order

event_mins = []
event_maxs = []
for i in range(len(candidate_events)): # find oldest and youngest age of event within CDF_stratigraphic_constraint_threshold, used to constrain stratigraphic order below
    indices = [i for i, val in enumerate(candidate_events[i].CDF) if CDF_stratigraphic_constraint_threshold < val < 1 - CDF_stratigraphic_constraint_threshold]
    event_mins.append(min(np.array(candidate_events[i].age)[indices]))
    event_maxs.append(max(np.array(candidate_events[i].age)[indices]))

#Create model
model = pl.LpProblem("Optimized Paleoseismic History", pl.LpMinimize)

#Create n binary decision variables for each candidate event, each representing a different combination of rupture endpoints (n = number of valid combinations)

y = []
for n in range(len(candidate_events)):
    var_set = pl.LpVariable.dicts(f"y_{n}", range(len(trench_displacements[n][0])), cat='Binary')
    y.append(var_set)

#Constraint: trench events stay in stratigraphic order

def natural_sort_key(s):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r'(\d+)', s)
    ]

for site in included_site_names:

    event_names = []

    for event in master_event_names:
        if event.split('_')[0] == site:
            event_names.append(event)
    #Add special condition for Pallett Creek because the oldest event is the earliest letter in the alphabet
    if site == 'Pallett':
        event_names_sorted = ['Pallett_Event_V', 'Pallett_Event_T_and_Hiatus', 'Pallett_Event_R', 'Pallett_Event_N', 'Pallett_Event_I', 'Pallett_Event_F_and_Hiatus', 'Pallett_Event_D', 'Pallett_Event_C']
   #Add special condition for Elizabeth Lake because Event X is chronologically first but falls last when sorted
    elif site == 'Elizabeth':
        event_names_sorted = ['Elizabeth_Event_ELX', 'Elizabeth_Event_EL2', 'Elizabeth_Event_EL3', 'Elizabeth_Event_EL4']
    else:
        event_names_sorted = sorted(event_names, key=natural_sort_key)

    #Compare each event with the event immediately below it
    for j in range(len(event_names_sorted) - 1):

        name1 = event_names_sorted[j]
        name2 = event_names_sorted[j + 1]

        #Find the candidate earthquake indices belonging to each event
        group1 = candidates_df.index[
            candidates_df[name1] == 1
        ]

        group2 = candidates_df.index[
            candidates_df[name2] == 1
        ]

        #Multiply each event's modeled age for that candidate event by its decision variable for that event
        event1_min = pl.lpSum(
            event_mins[n] * y[n][i]
            for n in group1
            for i in range(len(y[n]))
        )

        event2_min = pl.lpSum(
            event_mins[n] * y[n][i]
            for n in group2
            for i in range(len(y[n]))
        )

        event1_max = pl.lpSum(
            event_maxs[n] * y[n][i]
            for n in group1
            for i in range(len(y[n]))
        )
 
        event2_max = pl.lpSum(
            event_maxs[n] * y[n][i]
            for n in group2
            for i in range(len(y[n]))
        )

#WARNING: THIS LOGIC WILL BREAK IF AN EVENT'S AGE RANGE OVERLAPS WITH ANOTHER EVENT FROM THE SAME TRENCH (i.e. at Elizabeth Lake...)
#This constraint will make model infeasible if set too low with sites in the model that have overlapping ages (i.e. one event is much better constrained than another)
        #Must use greater than or equal to for continuous optimization
#        model += event1_min >= event2_min
#        model += event1_max >= event2_max

#Constraint: One event per column (each trench event must belong to only one candidate event)

# Constraint: 1 event per column, EXCEPT in the cases of 'event and hiatus', where we don't constrain how many events can be within hiatus
for i in range(len(master_event_names)):
    event_name = master_event_names[i]
    if any(string == 'Hiatus' for string in event_name.split('_')): # don't constrain if this is an event and hiatus
        continue
    else:
        group = candidates_df[candidates_df[event_name] == 1].index
        if len(group) > 0:
            model += pl.lpSum(y[n][i] for n in group for i in range(len(y[n]))) == 1
            

#Constraint: Pick AT MOST one endpoint from a candidate event's displacement set (may be redundant, not tested)
for n in range(len(candidate_events)):
    model += pl.lpSum(y[n][i] for i in range(len(y[n]))) <= 1

#Constraint: If trench showed 'ruptured' (1), displacement must be greater than displacement threshold (and vice versa)

#Go through all the displacement subarrays that correspond to trenches that are ruptured (and unruptured) and force the y[n][i] that correspond to
#displacements in these subarrays that are not consistent with either being observed or not observed to be 0

for n in range(len(candidates_df)):
    candidate = candidate_events[n]
    ruptured_sites = [event.name.split('_')[0] for i, event in enumerate(candidate.event_list) if event.name in master_event_names and event.name.split('_')[-1] != 'Hiatus'] # return the names of the sites that were visibly ruptured in this candidate; in other words, the sites for which we will force displacements to be above threshold, but NOT event and hiatuses
    all_sites = [object.name.split('_')[0] for object in candidate.event_list] # return the names of all sites involved in this event, through hiatuses, basal boundaries, or events
    nonruptured_sites = [site for site in included_site_names if site not in all_sites] # return the names of the sites that were not visibly ruptured and were not within a hiatus or basal boundary at the time of the event (which would negate displacement constraints); in other words, the sites for which we will force displacements to be below threshold
    for site in ruptured_sites:
        site_idx = included_site_names.index(site)
        displacements = np.array(trench_displacements[n][site_idx], dtype=object)
        indices = np.where(displacements < threshold[site_idx])[0] # find the indices inside trench_displacements for the ruptured sites where the displacements would be too small to be observed and force the model not to choose these endpoint combinations (set those ys == 0)
        for index in indices:
            model += y[n][index] == 0
    for site in nonruptured_sites:
        site_idx = included_site_names.index(site)
        displacements = np.array(trench_displacements[n][site_idx], dtype=object)
        indices = np.where(displacements > threshold[site_idx])[0] # find the indices inside trench_displacements for the ruptured sites where the displacements would be too large to be not observed and force the model not to choose these endpoint combinations (set those ys == 0)
        for index in indices:
            model += y[n][index] == 0

C:\Users\erikc\anaconda3\Lib\site-packages\pulp\pulp.py:1717: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


In [ ]:
#ILP Model

#Settings:

#Choose whether to turn on slip rate optimization
SRS_optimize = 'on'

#If SRS_optimize on, choose slip rate scenario: 'max', 'min', or 'mean' (max = use upper bound on displacement or slip rate for each site, min = use lower bound on displacement or slip rate for each site)
SRS_scenario = 'mean'

#Choose whether to turn on slip-per-event constraints, slip-per-event optimization, or neither
SPE_constrain = 'off'
SPE_optimize = 'on'

#If SPE_optimize on, choose slip-per-event scenario: 'max', 'min', or 'mean' (max = use upper bound on slip per event, min = use lower bound on slip per event)
SPE_scenario = 'mean'

#Choose whether to constrain event size based on estimates of event slip relative to other events at the same site
SPE_relative_constrain = 'on'

equality_threshold = 1 #Maximum allowed difference in lateral slip (in meters) to be allowed to be 'relatively' the same sized event

#Choose weighting of residuals for slip rate and slip-per-event estimations: default is no weighting (i.e. SRS_weight = 1, SPE_weight = 1)
#*When no weighting is chosen, model will by nature tend to place more importance on SRS residuals because deviations tend to be larger there

SRS_weight = 1 #The higher the weighting, the more importance the model places on optimizing this parameter
SPE_weight = 1

##(former position of ILP Constraints)

#Modeled Slip using endpoint decision variables and displacements matrix (all sites for a candidate event have same dimension of y[n], which is the same dimension as displacements[n][site]):

#Slip rate data: if slip rate data lower bound is younger than its two adjacent paleoseismic sites, use its displacement + time interval
#if slip rate data lower bound is older than its two adjacent paleoseismic sites, only use the calculated rate x time interval between oldest adjacent earthquake and 1812/1857

#Create objective functions for each SRS that was chosen to optimize with

residual = pl.LpAffineExpression()

#Replaced age_limits everywhere below with trench_oldest_ages, since using age_limits means there's
#a good chance we aren't constraining all the events that involve those trenches. However, this does add some risk
#that we are constraining an event that ruptured through before the oldest event at a site (for a given scenario)

trench_location = [site.position for site in included_sites] # store array of trench locations for use in code below

#MEAN slip scenario
if SRS_optimize == 'on':
    if SRS_scenario == 'mean':
        for site in optimize_site_names:
            r = pl.LpVariable(f"SRS_r_{site}", lowBound=0) #Create residual variable for each slip rate site to optimize with
            index = np.where(SRS_site_names == site)[0][0]
            position = SRS_df.loc[SRS_df['name'] == site, 'location']
            N_trench_index = np.searchsorted(trench_location, position) - 1 #Find the indexes of the two adjacent trenches
            S_trench_index = N_trench_index + 1
        #For this study, I have chosen to use the youngest of the two adjacent paleoseismic site's oldest ages,
        #to make sure that the displacement is constrained using ALL the earthquakes that have ruptured through
        #the slip rate site. Another possibility is to use min() instead, since we have dealt with the censoring problem
            if isinstance(N_trench_index, np.int64): #Handles bug with N_trench_index alternating data types
                adjacent_sites_age_limit = max(int(included_sites[N_trench_index].basal_boundary), int(included_sites[S_trench_index].basal_boundary))
            else:
                adjacent_sites_age_limit = max(int(included_sites[N_trench_index[0]].basal_boundary), int(included_sites[S_trench_index[0]].basal_boundary))
            lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
            #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
            if lower_bound <= adjacent_sites_age_limit:
                lower_bound = adjacent_sites_age_limit
            upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
            constrained_event_indices = []
            for idx in range(len(candidate_events)):
                if any(lower_bound < year < upper_bound for year in [year for i, year in enumerate(candidate_events[idx].age) if CORRELATION_THRESHOLD < candidate_events[idx].CDF[i] < 1 - CORRELATION_THRESHOLD]): # see if event has any time interval within the time constrained by upper and lower bound where its CDF is within the correlation threshold
                    constrained_event_indices.append(idx)
            #If lower age on displacement measurement is within paleoseismic record, constrain events in that time period to that displacement
            if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15: #Add 15 year buffer (for Bidart)
                model += r >= SRS_weight * ((SRS_df.loc[SRS_df['name'] == site, 'displacement mean']) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                model += r >= SRS_weight * (-((SRS_df.loc[SRS_df['name'] == site, 'displacement mean']) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))))
                residual += r
            #If lower age on displacement measurement is pre-record, turn it into a slip rate to optimize against
            else:
                slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement mean'])/(SRS_df.loc[SRS_df['name'] == site, 'age upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'age lower bound'])
                model += r >= SRS_weight * (slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                model += r >= SRS_weight * (-(slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))))
                residual += r

#MAX slip scenario

    if SRS_scenario == 'max':
        for site in optimize_site_names:
            r = pl.LpVariable(f"SRS_r_{site}", lowBound=0) #Create residual variable for each slip rate site to optimize with
            index = np.where(SRS_site_names == site)[0][0]
            position = SRS_df.loc[SRS_df['name'] == site, 'location']
            N_trench_index = np.searchsorted(trench_location, position) - 1
            S_trench_index = N_trench_index + 1
        #For this study, I have chosen to use the youngest of the two adjacent paleoseismic site's oldest ages,
        #to make sure that the displacement is constrained using ALL the earthquakes that have ruptured through
        #the slip rate site. Another possibility is to use min() instead, since we have dealt with the censoring problem
            if isinstance(N_trench_index, np.int64):
                adjacent_sites_age_limit = max(int(included_sites[N_trench_index].basal_boundary), int(included_sites[S_trench_index].basal_boundary))
            else:
                adjacent_sites_age_limit = max(int(included_sites[N_trench_index[0]].basal_boundary), int(included_sites[S_trench_index[0]].basal_boundary))
            lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
        #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
            if lower_bound <= adjacent_sites_age_limit:
                lower_bound = adjacent_sites_age_limit
            upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
            constrained_event_indices = []
            for idx in range(len(candidate_events)):
                if any(lower_bound < year < upper_bound for year in [year for i, year in enumerate(candidate_events[idx].age) if CORRELATION_THRESHOLD < candidate_events[idx].CDF[i] < 1 - CORRELATION_THRESHOLD]): # see if event has any time interval within the time constrained by upper and lower bound where its CDF is within the correlation threshold
                    constrained_event_indices.append(idx)
            if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15: #Add 15 year buffer (for Bidart)
                model += r >= SRS_weight * (SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'] - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                model += r >= SRS_weight * (-(SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'] - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))))
                residual += r
            #If lower age on displacement measurement is pre-record, turn it into a slip rate to optimize against
            else:
                slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/(SRS_df.loc[SRS_df['name'] == site, 'oldest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'youngest lower bound'])
                model += r >= SRS_weight * (slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                model += r >= SRS_weight * (-(slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))))
                residual += r

#MIN slip scenario

    if SRS_scenario == 'min':
        for site in optimize_site_names:
            r = pl.LpVariable(f"SRS_r_{site}", lowBound=0) #Create residual variable for each slip rate site to optimize with
            index = np.where(SRS_site_names == site)[0][0]
            position = SRS_df.loc[SRS_df['name'] == site, 'location']
            N_trench_index = np.searchsorted(trench_location, position) - 1
            S_trench_index = N_trench_index + 1
        #For this study, I have chosen to use the youngest of the two adjacent paleoseismic site's oldest ages,
        #to make sure that the displacement is constrained using ALL the earthquakes that have ruptured through
        #the slip rate site. Another possibility is to use min() instead, since we have dealt with the censoring problem
            if isinstance(N_trench_index, np.int64):
                adjacent_sites_age_limit = max(int(included_sites[N_trench_index].basal_boundary), int(included_sites[S_trench_index].basal_boundary))
            else:
                adjacent_sites_age_limit = max(int(included_sites[N_trench_index[0]].basal_boundary), int(included_sites[S_trench_index[0]].basal_boundary))
            lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
        #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
            if lower_bound <= adjacent_sites_age_limit:
                lower_bound = adjacent_sites_age_limit
            upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
            constrained_event_indices = []
            for idx in range(len(candidate_events)):
                if any(lower_bound < year < upper_bound for year in [year for i, year in enumerate(candidate_events[idx].age) if CORRELATION_THRESHOLD < candidate_events[idx].CDF[i] < 1 - CORRELATION_THRESHOLD]): # see if event has any time interval within the time constrained by upper and lower bound where its CDF is within the correlation threshold
                    constrained_event_indices.append(idx)
            if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15: #Add 15 year buffer (for Bidart)
                model += r >= SRS_weight * (SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                model += r >= SRS_weight * (-(SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))))
                residual += r
            #If lower age on displacement measurement is pre-record, turn it into a slip rate to optimize against
            else:
                slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'])/(SRS_df.loc[SRS_df['name'] == site, 'youngest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'oldest lower bound'])
                model += r >= SRS_weight * (slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))))
                model += r >= SRS_weight * (-(slip_rate*(upper_bound - lower_bound) - pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))))
                residual += r

#Create constraints for each SRS that was chosen to constrain (upper and lower bound) with

for site in constrain_site_names:
    index = np.where(SRS_site_names == site)[0][0]
    position = SRS_df.loc[SRS_df['name'] == site, 'location']
    N_trench_index = np.searchsorted(trench_location, position) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(included_sites[N_trench_index].basal_boundary), int(included_sites[S_trench_index].basal_boundary))
    else:
        adjacent_sites_age_limit = max(int(included_sites[N_trench_index[0]].basal_boundary), int(included_sites[S_trench_index[0]].basal_boundary))
    lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
    #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
    if lower_bound <= adjacent_sites_age_limit:
        lower_bound = adjacent_sites_age_limit
    upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
    constrained_event_indices = []
    for idx in range(len(candidate_events)):
        if any(lower_bound < year < upper_bound for year in [year for i, year in enumerate(candidate_events[idx].age) if CORRELATION_THRESHOLD < candidate_events[idx].CDF[i] < 1 - CORRELATION_THRESHOLD]): # see if event has any time interval within the time constrained by upper and lower bound where its CDF is within the correlation threshold
            constrained_event_indices.append(idx)
    if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15:
        model += SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'].item() <= pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))) <= SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'].item()
    else:
        low_slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'])/(SRS_df.loc[SRS_df['name'] == site, 'youngest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'oldest lower bound'])
        high_slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/(SRS_df.loc[SRS_df['name'] == site, 'oldest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'youngest lower bound'])
        model += (low_slip_rate.item()*(upper_bound - lower_bound)) <= pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))) <= (high_slip_rate.item()*(upper_bound - lower_bound))

#Create constraints for each SRS that was chosen to constrain (lower bound) with

for site in constrain_lower_bd_site_names:
    index = np.where(SRS_site_names == site)[0][0]
    position = SRS_df.loc[SRS_df['name'] == site, 'location']
    N_trench_index = np.searchsorted(trench_location, position) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(included_sites[N_trench_index].basal_boundary), int(included_sites[S_trench_index].basal_boundary))
    else:
        adjacent_sites_age_limit = max(int(included_sites[N_trench_index[0]].basal_boundary), int(included_sites[S_trench_index[0]].basal_boundary))
    lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
    #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
    if lower_bound <= adjacent_sites_age_limit:
        lower_bound = adjacent_sites_age_limit
    upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
    constrained_event_indices = []
    for idx in range(len(candidate_events)):
        if any(lower_bound < year < upper_bound for year in [year for i, year in enumerate(candidate_events[idx].age) if CORRELATION_THRESHOLD < candidate_events[idx].CDF[i] < 1 - CORRELATION_THRESHOLD]): # see if event has any time interval within the time constrained by upper and lower bound where its CDF is within the correlation threshold
            constrained_event_indices.append(idx)
    if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15:
        model += SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'] <= pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))
    else:
        low_slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement lower bound'])/(SRS_df.loc[SRS_df['name'] == site, 'youngest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'oldest lower bound'])
        model += (low_slip_rate.item()*(upper_bound - lower_bound)) <= pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0])))

#Create constraints for each SRS that was chosen to constrain (upper bound) with

for site in constrain_upper_bd_site_names:
    index = np.where(SRS_site_names == site)[0][0]
    position = SRS_df.loc[SRS_df['name'] == site, 'location']
    N_trench_index = np.searchsorted(trench_location, position) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64):
        adjacent_sites_age_limit = max(int(included_sites[N_trench_index].basal_boundary), int(included_sites[S_trench_index].basal_boundary))
    else:
        adjacent_sites_age_limit = max(int(included_sites[N_trench_index[0]].basal_boundary), int(included_sites[S_trench_index[0]].basal_boundary))
    lower_bound = (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item()
    #Set condition to make sure the events we are constraining happened when we have paleoseismic records on both sides
    if lower_bound <= adjacent_sites_age_limit:
        lower_bound = adjacent_sites_age_limit
    upper_bound = SRS_df.loc[SRS_df['name'] == site, 'age upper bound'].item()
    constrained_event_indices = []
    for idx in range(len(candidate_events)):
        if any(lower_bound < year < upper_bound for year in [year for i, year in enumerate(candidate_events[idx].age) if CORRELATION_THRESHOLD < candidate_events[idx].CDF[i] < 1 - CORRELATION_THRESHOLD]): # see if event has any time interval within the time constrained by upper and lower bound where its CDF is within the correlation threshold
            constrained_event_indices.append(idx)
    if (SRS_df.loc[SRS_df['name'] == site, 'age lower bound']).item() > adjacent_sites_age_limit - 15:
        model += pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))) <= SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound']
    else:
        high_slip_rate = (SRS_df.loc[SRS_df['name'] == site, 'displacement upper bound'])/(SRS_df.loc[SRS_df['name'] == site, 'oldest upper bound'] - SRS_df.loc[SRS_df['name'] == site, 'youngest lower bound'])
        model += pl.lpSum(srs_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(srs_displacements[n][0]))) <= (high_slip_rate.item()*(upper_bound - lower_bound))


        
#Optimize based on slip-per-event estimations
if SPE_optimize == 'on':
    if SPE_scenario == 'mean': #Optimize based on mean estimated slip per event
        for event in SPE_df['event']: 
            r = pl.LpVariable(f"SPE_r_{event}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0]
            site = event.split('_')[0]
            index = included_site_names.index(site)
#Create objective functions for all displacements associated with that event at that site
            model += r >= SPE_weight * (SPE_df.loc[SPE_df['event'] == event, 'mean'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
            model += r >= SPE_weight * (-(SPE_df.loc[SPE_df['event'] == event, 'mean'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))))
            residual += r
    if SPE_scenario == 'max': #Optimize based on max estimated slip per event
        for event in SPE_df['event']: 
            r = pl.LpVariable(f"SPE_r_{event}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0]
            site = event.split('_')[0]
            index = included_site_names.index(site)
#Create objective functions for all displacements associated with that event at that site
            model += r >= SPE_weight * (SPE_df.loc[SPE_df['event'] == event, 'upper_bd'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
            model += r >= SPE_weight * (-(SPE_df.loc[SPE_df['event'] == event, 'upper_bd'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))))
            residual += r
    if SPE_scenario == 'min': #Optimize based on min estimated slip per event
        for event in SPE_df['event']: 
            r = pl.LpVariable(f"SPE_r_{event}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0]
            site = event.split('_')[0]
            index = included_site_names.index(site)
#Create objective functions for all displacements associated with that event at that site
            model += r >= SPE_weight * (SPE_df.loc[SPE_df['event'] == event, 'lower_bd'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
            model += r >= SPE_weight * (-(SPE_df.loc[SPE_df['event'] == event, 'lower_bd'] - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))))
            residual += r

#Optimize based on slip-per-event-group estimations
if SPE_optimize == 'on':
    if SPE_scenario == 'mean':
        for group in summed_d_events: 
                r = pl.LpVariable(f"SPE_r_{group}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
                n_indices = []
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
                for i in range(len(group)):
                    n_indices.append(np.where(candidates_df[group[i]] == 1)[0])
                n_indices = [item for sublist in n_indices for item in sublist]
                site = group[0].split('_')[0]
                index = included_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
                model += r >= SPE_weight * (summed_d_mean - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
                model += r >= SPE_weight * (-(summed_d_mean - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))))
                residual += r
    if SPE_scenario == 'min':
        for group in summed_d_events: 
                r = pl.LpVariable(f"SPE_r_{group}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
                n_indices = []
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
                for i in range(len(group)):
                    n_indices.append(np.where(candidates_df[group[i]] == 1)[0])
                n_indices = [item for sublist in n_indices for item in sublist]
                site = group[0].split('_')[0]
                index = included_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
                model += r >= SPE_weight * (summed_d_lower_bd - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
                model += r >= SPE_weight * (-(summed_d_lower_bd - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))))
                residual += r
    if SPE_scenario == 'max':
        for group in summed_d_events: 
                r = pl.LpVariable(f"SPE_r_{group}", lowBound=0) #Create residual variable for each event to optimize with
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
                n_indices = []
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
                for i in range(len(group)):
                    n_indices.append(np.where(candidates_df[group[i]] == 1)[0])
                n_indices = [item for sublist in n_indices for item in sublist]
                site = group[0].split('_')[0]
                index = included_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
                model += r >= SPE_weight * (summed_d_upper_bd - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))))
                model += r >= SPE_weight * (-(summed_d_upper_bd - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0])))))
                residual += r


#Constrain based on slip-per-event-group estimations
if SPE_constrain == 'on':
    for group in summed_d_events: 
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
        n_indices = []
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
        for i in range(len(group)):
            n_indices.append(np.where(candidates_df[group[i]] == 1)[0])
        n_indices = [item for sublist in n_indices for item in sublist]
        site = group[0].split('_')[0]
        index = included_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
        model += summed_d_lower_bd <= pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))) <= summed_d_upper_bd


#Create slip-per-event constraints
if SPE_constrain == 'on':
    for event in SPE_df['event']: 
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
        n_indices = np.where(candidates_df[event] == 1)[0]
        site = event.split('_')[0]
        index = included_site_names.index(site)
#Create objective functions for all displacement matrices associated with that site and those candidate events
        model += SPE_df.loc[SPE_df['event'] == event, 'lower_bd'].item() <= pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))) <= SPE_df.loc[SPE_df['event'] == event, 'upper_bd'].item()

#Constrain slip using the default slip rate bounds for points of chosen spacing along fault
for location in default_slip_rate_locations:
    N_trench_index = np.searchsorted(trench_location, location) - 1
    S_trench_index = N_trench_index + 1
    if isinstance(N_trench_index, np.int64): #Handles bug with N_trench_index alternating data types
        adjacent_sites_age_limit = max(int(included_sites[N_trench_index].basal_boundary), int(included_sites[S_trench_index].basal_boundary))
    else:
        adjacent_sites_age_limit = max(int(included_sites[N_trench_index[0]].basal_boundary), int(included_sites[S_trench_index[0]].basal_boundary))
    constrained_event_indices = []
    for idx in range(len(candidate_events)):
        if any(adjacent_sites_age_limit < year for year in [year for i, year in enumerate(candidate_events[idx].age) if CORRELATION_THRESHOLD < candidate_events[idx].CDF[i] < 1 - CORRELATION_THRESHOLD]): # see if event has any time interval within the time constrained by upper and lower bound where its CDF is within the correlation threshold
            constrained_event_indices.append(idx)    
    index = np.where(default_slip_rate_locations == location)[0][0]
    #Put different default slip constraints on different sections of the fault
###IMPORTANT DECISION: to deal with problems resulting low slip rates at fault tips from the finite nature of the model, only constrain locations >20 km from the N end of the model (slip naturally tapers near San Gorgonio Pass)
    if 20 < location < 315:
        model += Big_Bend_default_high_rate*(1812 - adjacent_sites_age_limit) >= pl.lpSum(default_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(default_displacements[n][0]))) >= Big_Bend_default_low_rate*((1812 - adjacent_sites_age_limit))
    if location > 315:
        model += San_Bernadino_default_high_rate*(1812 - adjacent_sites_age_limit) >= pl.lpSum(default_displacements[n][index][i] * y[n][i] for n in constrained_event_indices for i in range(len(default_displacements[n][0]))) >= San_Bernadino_default_low_rate*((1812 - adjacent_sites_age_limit))


#Constrain event size compared to other events at that same site (if SPE_relative_constrain == 'on')
if SPE_relative_constrain == 'on':
    for site_index in range(len(SPE_relative_list)):
        for event_index in range(len(SPE_relative_list[site_index]) - 1):
            event = SPE_relative_list[site_index][event_index]
            next_event = SPE_relative_list[site_index][event_index + 1]
#Find which candidate events (which 'n') correspond to each trench event and find the displacements matrix that corresponds to the given site
            n_indices = np.where(candidates_df[event] == 1)[0] #Indices of the first event to be compared
            o_indices = np.where(candidates_df[next_event] == 1)[0] #Indices of the next event (larger event, according to arbitrary decision on how dataframe should be ordered) 
            site = event.split('_')[0]
            index = included_site_names.index(site)
#Create objective functions for all displacement matrices associated with that site and those candidate events
            model += pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices for i in range(len(trench_displacements[n][0]))) <= pl.lpSum(trench_displacements[o][index][i] * y[o][i] for o in o_indices for i in range(len(trench_displacements[o][0])))


#Constrain event size to be comparable to other similar-sized events at that same site (within 'equality_threshold' meters, which is arbitrary measurement) (if SPE_relative_constrain == 'on')
if SPE_relative_constrain == 'on':
    for group in equal_size_events:
        for idx in range(len(group) - 1): #Loop through each pair in the group of similar sized events to constrain displacement difference to be within equality_threshold
#Find which candidate events (which 'n') correspond to each trench event in the pair and find the displacements matrix that corresponds to the given site
            n_indices = [[], []]
#Go through each slip-per-event group and save the indices of the candidate events that each trench event belongs to
            n_indices[0].append(np.where(candidates_df[group[idx]] == 1)[0])
            n_indices[1].append(np.where(candidates_df[group[idx + 1]] == 1)[0])
        #n_indices = [item for sublist in n_indices for item in sublist]
            site = group[0].split('_')[0]
            index = included_site_names.index(site)
#Create objective functions for all displacements associated with that group at that site
            model += - equality_threshold <= pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices[0][0] for i in range(len(trench_displacements[n][0]))) - pl.lpSum(trench_displacements[n][index][i] * y[n][i] for n in n_indices[1][0] for i in range(len(trench_displacements[n][0]))) <= equality_threshold



#Solve
model += residual


#Selection: choose wether to return single best-fitting history or family of best-fitting histories

Single_solution = False #Select single best event history vs. multiple simiarly-fitting histories
Solution_count = 10 #Choose how many optimal histories to keep (default = 10)

if Single_solution == True:
    solver = pl.GUROBI(
    msg=True,
    PoolSearchMode=0,
    MIPGap=0.01,
    TimeLimit=600,
    Symmetry=0
    )
    model.solve(solver)
if Single_solution == False:
    solver = pl.GUROBI(
    msg=True,
    PoolSearchMode=2,
    PoolSolutions=Solution_count,
    PoolGap=0.02,
    MIPGap=0.01,
    TimeLimit=30,
    Symmetry=0
    )
    model.solve(solver)
    gmodel = model.solverModel
    #solver.buildSolverModel(model)
    #gmodel = solver.solverModel
    gmodel.optimize()
    
#^Params:
# Report live updates from model while running
# 2 = search for 'n' best solutions, 1 = look for optimum and keep alternative solutions, 0 = only search for single optimum solution
# Keep up to 200 solutions from solutions that satisfy PoolGap constraint
# Keep solutions that are within this percentage (100 x val) difference from the optimal solution (BestBd): in other words, keep solutions with <= this 'Gap'
# Stop once solution falls below this percentage (100 x val) difference from the optimal solution (BestBd): in other words, stop once 'Gap' falls below this
# Stop model if solution is not found by this time limit (in s)
# How aggresively to break symmetry: in other words, how aggresively to cut branches in model with many similar solutions (from 0 = off to 2 = aggressive)

#Print off the candidate events of the best history that the model chooses
chosen_endpoints = []
for i in range(len(candidate_events)):
    for n in y[i]:
        if pl.value(y[i][n]) == 1:
            chosen_endpoints.append(y[i][n])

#Print out chosen endpoint sets and minimized d (absolute deviation)
print("Chosen endpoints:", chosen_endpoints)
print("Selected total deviation:", pl.value(model.objective))

Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 10
Set parameter PoolGap to value 0.02
Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 30
Set parameter Symmetry to value 0


C:\Users\erikc\anaconda3\Lib\site-packages\pulp\pulp.py:2189: UserWarning: Overwriting previously set objective.
  warnings.warn("Overwriting previously set objective.")


Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) 3 100U, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  30
MIPGap  0.01
Symmetry  0
PoolSearchMode  2
PoolGap  0.02

Optimize a model with 602473 rows, 199321 columns and 8681637 nonzeros (Min)
Model fingerprint: 0x01fc24aa
Model has 32 linear objective coefficients
Variable types: 46 continuous, 199275 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-01, 6e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-01, 8e+01]

Presolve removed 601404 rows and 188616 columns
Presolve time: 3.81s
Presolved: 1069 rows, 10705 columns, 737926 nonzeros
Variable types: 46 continuous, 10659 integer (10659 binary)
Performing another presolve...
Presolve removed 887 rows and 4147 columns
Presolve time: 2.71s

Root simplex log...

Iterati

In [31]:
#Infeasbility Problem
# Attempt 1: Commented out relative size and equal size constraints in SRS and SPE dataframes cell and in ILP Model and comment out new stratigraphic order constraints in ILP Model constraints cell
# Attempt 2: Model ran: infeasibility is caused by one of these two things: add back relative size and equal size constraints first to test
# Attempt 3: Model ran: infeasibility is caused by new stratigraphic order code: increase stratigraphic_order_CDF_threshold until model works
# Attempt 4: Model ran: infeasibility was caused by leaving the stratigraphic_order_CDF_threshold too low (0.1) - runs for 0.4, test at 0.3
# Attempt 5: Model ran: however, it has selected 48 events, likely because of this constraint. See if reverting back to 0.4 makes the event count more reasonable
# Attempt 6: Model still chose 47 events, way too high. Testing for 0.48. If this doesn't give a reasonable solution (~ 22 events), may have to rerun model after clearing kernel to see if this result is the effect of overconstraining with the percentile constraint or an error with displacement matrices calculation
# Attempt 7: Model still chose 46 events at 0.45. Testing rerunning model with no stratigraphic order constraint
# Attempt 8: Model still ran but chose 51 events now. Issue is seemingly not stemming from our stratigraphic order constraints: test turning SPE constrain 'on'
# Attempt 9: Testing SPE_optimize = 'on' just in case: also didn't fix problem, still choosing too many events. Look at displacement matrices

In [ ]:
len(chosen_endpoints)

In [ ]:
#Try running above code to test if model will succesfully run

In [87]:
for event in master_event_names:
    if sum(candidates_df[event]) == 1:
        print(event)

Frazier_Event_9
Burro_Event_9


In [ ]:
#Paste in cells from thesis condensed and try to get all cells to work

In [ ]:
#Make duplicate of notebook and synthesize with 'Thesis Condensed to Coachella' to get new results through San Gorgonio Pass to Indio Strand

In [ ]:
##

In [725]:
# Set up master event list for enumeration of candidates

included_sites = [Annette, Frazier, Pallett]
included_site_names = [site.site_name for site in included_sites]

LEFT_END = 0
LEFT_END_MINSLIP = 0
RIGHT_END = 390
RIGHT_END_MINSLIP = 0

master_event_list = [] # earthquake objects
master_site_list = [] # paleoseismic site objects in fault-position order

for site in included_sites:
    master_event_list = master_event_list + site.event_list()
    master_site_list = master_site_list + [site] * len(site.event_list())

master_event_names = [eq.name for eq in master_event_list] # corresponding list of event names
master_site_names = [site.site_name for site in master_site_list] # corresponding list of event names

#master_event_names.index(Pallett.events[7].name) Example of how to find index of an event in matrix

total_events = len(master_event_names)

# Construct correlation matrix
correlation_matrix = np.zeros((total_events,total_events))

for i in range(total_events):
    for j in range(total_events):
        correlation_matrix[i,j] = master_event_list[i].BC(master_event_list[j])

# Recursive candidate enumeration algorithm
candidate_events = []
RIGHT = 1
LEFT = 2
def make_candidate(correlation_row, new_event, direction = RIGHT, left_events = [], right_events=[], right_end = None, right_minslip = None):
    # correlation_row: one row of the correlation matrix
    # root: event at the root of this candidate tree. All events correlate to root by definition.
    # left_events: list of events in the left dirction already added to the candidate
    # right_events: list of events in the right direction already added to the candidate
    # new_event: proposed new event at the next trench site
    # direction: are we adding an event to the RIGHT or LEFT?
    
    # STEP 1: Check new event against all right and left events to make sure that this candidate is valid.
    # If not valid, then return, otherwise continue and make event and recursively call as needed.
    new_event_index = master_event_names.index(new_event.name)
    combined_events = left_events + right_events
    for event in combined_events:
        event_index = master_event_names.index(event.name)
        # Test if new event is valid correlation to each other event in the list (except root).
        # If an invalid correlation is found, we return (end the function) and don't generate a candidate 
        if correlation_matrix[new_event_index,event_index] < CORRELATION_THRESHOLD: return

    # STEP 2. Look for the next possible event (one or more) and call make_candidate for each of these. This
    # step recursively builds the candidates. To ensure against infinite recursion:
    # If the direction == RIGHT, then we look to the right and then to the left
    # If the direction == LEFT, then we only look to the left
    if direction == RIGHT:
        # find name of rightmost site name
        rightmost_site_name = master_site_names[master_event_names.index(new_event.name)]
        # find the name of the next site or end of fault
        rightmost_site_index = included_site_names.index(rightmost_site_name)
        if rightmost_site_index == len(included_site_names) - 1:
            # We found the end of the fault, use this as the end point
            right_end = RIGHT_END
            right_minslip = RIGHT_END_MINSLIP
        else:
            # We look at the next trench site on the list and call next level of recursion if we find possible correlation
            next_trench_site = included_sites[rightmost_site_index + 1]
            next_trench_site_name = next_trench_site.site_name
            possible_correlative_event_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
            for event_index in possible_correlative_event_indices:
                # This is the key step in the recursion. If an event at the next trench has a possible correlation,
                # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
                # If more than one possible correlative is foumd make_candidate is called for each.
                if correlation_row[event_index] >= CORRELATION_THRESHOLD:
                    next_new_event = master_event_list[event_index]
                    make_candidate(correlation_row, next_new_event, RIGHT, left_events, right_events + [new_event])
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
            right_end = next_trench_site.position
            right_minslip = next_trench_site.slip_threshold

        # Final steps, looking to the right:
        right_events = right_events + [new_event] # Copy new event to right event list. List complete for this candidate.
        new_event = right_events[0] # new_event is now the root event so we can start recursion to the left
        
    # Finished looking to right, now look left
    
    leftmost_site_name = master_site_names[master_event_names.index(new_event.name)]
    # find the name of the next site or end of fault
    leftmost_site_index = included_site_names.index(leftmost_site_name)
    if leftmost_site_index == 0:
            # We found the end of the fault, use this as the other end point
        left_end = LEFT_END
        left_minslip = LEFT_END_MINSLIP
    else:
        # We look at the next trench site on the list and call next level of recursion if we find possible correlation
        next_trench_site = included_sites[leftmost_site_index - 1]
        next_trench_site_name = next_trench_site.site_name
        possible_correlative_event_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
        for event_index in possible_correlative_event_indices:
            # This is the key step in the recursion. If an event at the next trench has a possible correlation,
            # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
            # If more than one possible correlative is foumd make_candidate is called for each.
            if correlation_row[event_index] >= CORRELATION_THRESHOLD:
                next_new_event = master_event_list[event_index]
                make_candidate(correlation_row, next_new_event, LEFT, left_events + [new_event], right_events, right_end, right_minslip)
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
        left_end = next_trench_site.position
        left_minslip = next_trench_site.slip_threshold
    
    # Final steps, looking to the left:
    left_events = left_events + [new_event] # Copy new event to left event list. List complete for this candidate.
    left_events = left_events[1:] # Remove the root event left_events (it is preserced in right_events)
    temporary_event_name = right_events[0].name # name this event after the root event. Elaborate names later....

    # Final step: create candidate and add it to the list
    candidate_events.append(Candidate_Event(temporary_event_name, left_events[::-1] + right_events))
    candidate_events[-1].right_end = right_end
    candidate_events[-1].right_minslip = right_minslip
    candidate_events[-1].left_end = left_end
    candidate_events[-1].left_minslip = left_minslip
    

    



In [743]:
candidate_events = []
start_event = 8
make_candidate(correlation_matrix[start_event,:],master_event_list[start_event])
#correlation_matrix[6,:]

IndexError: list index out of range

In [729]:
for x in range(len(candidate_events)):
    print([event.name for event in candidate_events[x].event_list])
#indices = [i for i, x in enumerate(master_site_names) if x == 'Pallett']
#indices

In [81]:
for x in range(len(candidate_events)):
    print(candidate_events[x].left_end)

0
20
0
20
0
20


In [ ]:
## START OF ERIK EXPERIMENTING WITH STUFF

In [ ]:
#Edit preexisting master_event_list and event enumeration code to work with hiatuses

#Fundamental difference is that the single correlation matrix for events and hiatuses will be built with two different correlation techniques:
#event-to-event correlation will be decided based on Bhattacharya Coefficient (with the idea that different PDFs that represent the same event should
#overlap well, meaning their BC will be relatively high), whereas event-to-hiatus and hiatus-to-hiatus correlation will be conducted based on the overlap
#of their 95th percentile age range (this will work as long as we are still testing every possible pair in the string of pearls for correlation)
#(with the idea that a hiatus overlapping with an event or a hiatus overlapping with a hiatus does not imply anything about the likelihood of the event
#being correlated with the hiatus, only stating that it is possible that that hiatus concealed the given event)

In [ ]:
#Create correlation matrix to hold both events and hiatuses: instead of a function like .BC outputting a single number based
#on PDF agreement, the .CDF_overlap function will output only a 'TRUE' or 'FALSE', based on whether their x percentile ages overlap

